# L1/Q2 - Calibrated single-pass action-type decision on AndroidControl-Curated (T4)

State = goal (+ image), frozen `google/embeddinggemma-2` encoder + tiny linear head.
Compares **text-only vs multimodal** state, and **CE vs proper-scoring vs joint** (Q2).

Data: `batwBMW/AndroidControl_Curated` -> `benchmark_resource/android_control_high_bbox.json`
(8,435 bbox-corrected steps) + `android_control_images.tar.gz` (images). Images are also
cached locally on this Mac, but this notebook downloads from HF so it is self-contained.

Set Runtime/Accelerator to **GPU T4 x2** and enable **Internet** (required to download model+data).

In [ ]:
import torch
assert torch.cuda.is_available(), 'NO GPU - aborting before slow CPU load'
print('GPU OK:', torch.cuda.get_device_name(0))
import subprocess, sys
subprocess.run(['nvidia-smi'], check=False)

# deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers==5.19.0'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==6.1.0'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'], check=False)
import json, os, re, time, random, collections
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
save_dir = '/kaggle/working'

# ---- capture all stdout/stderr to a file in /kaggle/working so CLI can download it ----
import io
_log = open(os.path.join(save_dir, 'run.log'), 'a', buffering=1)
class _Tee(io.TextIOBase):
    def __init__(self, a, b): self.a, self.b = a, b
    def write(self, s): self.a.write(s); self.b.write(s); return len(s)
    def flush(self):
        self.a.flush(); self.b.flush()
sys.stdout = _Tee(sys.stdout, _log)
sys.stderr = _Tee(sys.stderr, _log)
import traceback
def _hook(tp, val, tb):
    traceback.print_exception(tp, val, tb)
    _log.flush()
    sys.__excepthook__(tp, val, tb)
sys.excepthook = _hook
print('logging redirected to run.log', flush=True)
print('imports OK')

In [ ]:
# ---- download labels + stream-extract only needed episodes (no token, disk-safe) ----
import urllib.request, tarfile, re, os
BASE = 'https://huggingface.co/datasets/batwBMW/AndroidControl_Curated/resolve/main'
def dl(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print('cached:', dest); return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print('downloading', dest)
    urllib.request.urlretrieve(url, dest)
    return dest

LAB = dl(BASE + '/benchmark_resource/android_control_high_bbox.json', os.path.join(save_dir,'android_control_high_bbox.json'))
rows = json.load(open(LAB))
LABEL_PAT = re.compile(r'^android_control_images/(\d+)/step_(\d+)\.png$')
def split_label(img):
    m = LABEL_PAT.match(img)
    return (m.group(1), m.group(2)) if m else (None, None)
episodes = set(split_label(r['image'])[0] for r in rows if split_label(r['image'])[0])
print('labels:', os.path.getsize(LAB), '| episodes needed:', len(episodes), flush=True)

# extract images OUTSIDE /kaggle/working so output stays small & downloadable
EXTRACT_ROOT = '/kaggle/tmp/ac_img'
os.makedirs(EXTRACT_ROOT, exist_ok=True)
IMG_TAR = dl(BASE + '/android_control_images.tar.gz', os.path.join(save_dir,'ac_imgs.tar.gz'))
print('extracting selected episodes...', flush=True)
TAR_PAT = re.compile(r'/(\d+)/step_(\d+)\.png$')
with tarfile.open(IMG_TAR) as tf:
    keep = []
    for mem in tf.getmembers():
        mm = TAR_PAT.search(mem.name)
        if mm and mm.group(1) in episodes:
            keep.append(mem)
    tf.extractall(EXTRACT_ROOT, members=keep)
print('extracted', len(keep), 'files', flush=True)
import glob
cands = glob.glob(os.path.join(EXTRACT_ROOT, '**', 'android_control_images'), recursive=True)
if cands:
    IMG_DIR = cands[0]
else:
    IMG_DIR = os.path.join(EXTRACT_ROOT, 'android_control_images')
print('IMG_DIR =', IMG_DIR, flush=True)
n_ok = 0
for r in rows:
    ep, st = split_label(r['image'])
    if ep and st:
        hits = glob.glob(os.path.join(EXTRACT_ROOT, '**', ep, 'step_' + st + '.png'), recursive=True)
        if hits:
            n_ok += 1
print('rows with local image:', n_ok, '/', len(rows), flush=True)


In [ ]:
# ---- build dataset: map curated image -> local extracted file, coarse action ----
rows = json.load(open(LAB))
def coarse(a):
    a = a or ''
    if a.startswith('input_text'): return 'input_text'
    if a.startswith('open_app'): return 'open_app'
    if a.startswith('swipe'): return 'swipe'
    if a.startswith('long_press'): return 'long_press'
    if a.startswith('navigate_back'): return 'navigate_back'
    if a.startswith('wait'): return 'wait'
    if a.startswith('scroll'): return 'scroll'
    if a.startswith('click'): return 'click'
    return 'other'

recs = []
for r in rows:
    m = re.match(r'android_control_images/(\d+)/step_(\d+)\.png', r['image'])
    if not m: continue
    ep, st = int(m.group(1)), int(m.group(2))
    p = f'{IMG_DIR}/{ep}/step_{st}.png'
    if not os.path.exists(p): continue
    recs.append(dict(episode_id=ep, step=st, goal=r.get('instruction') or '',
                      step_instruction=r.get('step_instruction') or '', image=p,
                      action=coarse(r['gt_action']), gt_action=r['gt_action']))
print('records with image:', len(recs))
print('action dist:', dict(collections.Counter(r['action'] for r in recs)))

ACTIONS = ['click', 'swipe', 'open_app', 'wait', 'input_text', 'navigate_back', 'long_press']
# split by episode
eps = collections.defaultdict(list)
for r in recs: eps[r['episode_id']].append(r)
eps = list(eps.items())
random.seed(123); random.shuffle(eps)
cut = int(len(eps)*0.8)
train = [r for ep,rs in dict(eps[:cut]).items() for r in rs]
test  = [r for ep,rs in dict(eps[cut:]).items() for r in rs]
print('episodes', len(eps), '| train', len(train), '| test', len(test))
save_dir = '/kaggle/working'
json.dump(train, open(save_dir+'/train.json','w'))
json.dump(test, open(save_dir+'/test.json','w'))

In [ ]:
# ---- load model + encode text-state and multimodal (frozen, OOM-safe) ----
PREFIX = 'task: classification | query: '
import traceback
t0 = time.time()
m = SentenceTransformer('google/embeddinggemma-2',
                        model_kwargs={'torch_dtype': torch.bfloat16},
                        config_kwargs={'audio_config': None})
m.eval().to('cuda')
print('model load', round(time.time()-t0,1), 's', flush=True)

def encode_text(recs):
    texts = [PREFIX + (r['goal'] or '') + ' [step: ' + (r['step_instruction'] or '') + ']' for r in recs]
    return np.asarray(m.encode(texts, convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)

def encode_mm(recs, bs=8):
    out = []
    n = len(recs)
    for i in range(0, n, bs):
        chunk = recs[i:i+bs]
        imgs = [Image.open(r['image']).convert('RGB').resize((224,224)) for r in chunk]
        inputs = [{'text': PREFIX + (r['goal'] or ''), 'image': im} for r,im in zip(chunk,imgs)]
        e = np.asarray(m.encode(inputs, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)
        out.append(e)
        if (i//bs) % 30 == 0:
            print('  encoded', min(i+bs, n), '/', n, flush=True)
            open(save_dir+'/progress.txt','a').write('mm '+str(min(i+bs,n))+'/'+str(n)+chr(10))
    return np.concatenate(out, axis=0)
train = json.load(open(save_dir+'/train.json'))
test  = json.load(open(save_dir+'/test.json'))
print('encoding text train/test...', flush=True)
S_tr_t = encode_text(train); S_te_t = encode_text(test)
np.save(save_dir+'/S_tr_t.npy', S_tr_t); np.save(save_dir+'/S_te_t.npy', S_te_t)
y_tr = np.array([ACTIONS.index(r['action']) if r['action'] in ACTIONS else ACTIONS.index('click') for r in train])
y_te = np.array([ACTIONS.index(r['action']) if r['action'] in ACTIONS else ACTIONS.index('click') for r in test])
np.save(save_dir+'/y_tr.npy', y_tr); np.save(save_dir+'/y_te.npy', y_te)
print('text emb', S_tr_t.shape, S_te_t.shape, 'done', round(time.time()-t0,1),'s', flush=True)

# multimodal: try, but ALWAYS produce a text result even if it fails
mm_ok = False
S_tr_m = S_te_m = None
try:
    print('encoding multimodal train/test...', flush=True)
    open(save_dir+'/progress.txt','w').write('mm start'+chr(10))
    S_tr_m = encode_mm(train, bs=8)
    S_te_m = encode_mm(test, bs=8)
    np.save(save_dir+'/S_tr_m.npy', S_tr_m); np.save(save_dir+'/S_te_m.npy', S_te_m)
    mm_ok = True
    print('multimodal emb', S_tr_m.shape, 'done', round(time.time()-t0,1),'s', flush=True)
except Exception as e:
    print('MULTIMODAL FAILED:', type(e).__name__, str(e), flush=True)
    traceback.print_exc()
    open(save_dir+'/mm_error.txt','w').write(repr(e))
    mm_ok = False
print('cached embeddings to', save_dir, '| mm_ok =', mm_ok, flush=True)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
def ece_binary(conf, correct, n_bins=10):
    n=len(conf)
    if n==0: return 0.0
    counts=[0]*n_bins; csum=[0.0]*n_bins; asum=[0.0]*n_bins
    for c,lab in zip(conf, correct):
        b=min(int(c*n_bins), n_bins-1)
        counts[b]+=1; csum[b]+=c; asum[b]+=1.0 if lab else 0.0
    tot=0.0
    for b in range(n_bins):
        if counts[b]==0: continue
        tot += (counts[b]/n)*abs(asum[b]/counts[b]-csum[b]/counts[b])
    return tot
def brier_binary(conf, correct):
    n=len(conf)
    return sum((c-(1.0 if lab else 0.0))**2 for c,lab in zip(conf,correct))/n

class Model(nn.Module):
    def __init__(self, dim, K):
        super().__init__()
        self.action = nn.Linear(dim, K)
        self.calib = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self, S):
        logits = self.action(S)
        probs = torch.softmax(logits, 1)
        t2 = torch.topk(probs,2,1).values
        margin = (t2[:,0]-t2[:,1]).unsqueeze(1)
        ci = torch.cat([probs.max(1,keepdim=True).values, margin], 1)
        return logits, torch.sigmoid(self.calib(ci)).squeeze(1)

def run(S_tr, S_te, y_tr, y_te, objective, epochs=30, lr=2e-3, label=''):
    S_tr=torch.from_numpy(S_tr).to('cuda'); S_te=torch.from_numpy(S_te).to('cuda')
    y_tr=torch.from_numpy(y_tr).to(torch.long).to('cuda'); y_te=torch.from_numpy(y_te).to(torch.long).to('cuda')
    K=len(ACTIONS); model=Model(S_tr.shape[1],K).to('cuda')
    opt=torch.optim.Adam(model.parameters(), lr=lr)
    ce=nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        logits, psucc = model(S_tr)
        correct = (logits.argmax(1)==y_tr).float()
        loss = ce(logits, y_tr)
        if objective in ('proper','joint'):
            w = 1.0
            if objective == 'joint':
                probs = torch.softmax(logits,1)
                mar = torch.topk(probs,2,1).values[:,0]-torch.topk(probs,2,1).values[:,1]
                w = 1.0 + (1.0-mar.detach())
            loss = loss + 0.5*torch.mean(w*(psucc-correct)**2)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        logits, psucc = model(S_te)
        pred = logits.argmax(1)
        acc = (pred==y_te).float().mean().item()
        correct = (pred==y_te).cpu().numpy().astype(np.float64)
        conf = psucc.cpu().numpy() if objective in ('proper','joint') else torch.softmax(logits,1).max(1).values.cpu().numpy()
        ece=ece_binary(conf,correct); bri=brier_binary(conf,correct)
    print(f'  [{label}|{objective}] acc={acc:.3f} ECE={ece:.4f} Brier={bri:.4f} conf={conf.mean():.3f}')
    return dict(mode=label, objective=objective, acc=round(acc,4), ece=round(ece,4), brier=round(bri,4))

S_tr_t=np.load(save_dir+'/S_tr_t.npy'); S_te_t=np.load(save_dir+'/S_te_t.npy')
y_tr=np.load(save_dir+'/y_tr.npy'); y_te=np.load(save_dir+'/y_te.npy')
mm_ok = os.path.exists(save_dir+'/S_tr_m.npy')
if mm_ok:
    S_tr_m=np.load(save_dir+'/S_tr_m.npy'); S_te_m=np.load(save_dir+'/S_te_m.npy')
    print('multimodal embeddings loaded, shape', S_tr_m.shape, flush=True)
else:
    print('multimodal embeddings NOT available - text-only run', flush=True)
results=[]
modes = [('text',S_tr_t,S_te_t)] + ([('multimodal',S_tr_m,S_te_m)] if mm_ok else [])
for mode, Str, Ste in modes:
    for obj in ('ce','proper','joint'):
        results.append(run(Str, Ste, y_tr, y_te, obj, label=mode))
print('=== SUMMARY ===')
import pandas as pd
pd.DataFrame(results).to_csv(save_dir+'/results.csv', index=False)
print(pd.DataFrame(results).to_string(index=False))

## How to read
- **acc** = action-type accuracy (baseline; frozen encoder + tiny head, `text` >> random 1/7).
- **multimodal vs text**: if `multimodal` acc/ECE improves over `text`, the image adds signal
  (this is the G2/G1 question settled with a trained head on full data).
- **CE vs proper vs joint** (same mode): Q2 — proper-scoring/joint should keep ECE lower than CE.

Results saved to `/kaggle/working/results.csv`. Copy that back to the research repo
`benchmarks/results/`.